### Autogen AgentChat

In [1]:
# Load environment variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# First concept: the Model
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [3]:
# Second concept: the Message
from autogen_agentchat.messages import TextMessage

message = TextMessage(content="I'd like to go to London", source="user")
message

TextMessage(id='ae42c13f-d961-47ea-bdb0-ed5beb75b80e', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 7, 19, 12, 48, 40, 43578, tzinfo=datetime.timezone.utc), content="I'd like to go to London", type='TextMessage')

In [4]:
# third concept: the Agent
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers.",
    model_client_stream=True
)

In [5]:
# Put it all together on-messages
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'Great choice! London has more history than your last family reunion! Ready to jet off, or do you want to know the best sights to see (other than the airport bathroom)?'

#### TOOLS FOR TICKET PRICES


In [6]:
# Let's make a local database of ticket prices
import os
import sqlite3

In [7]:
# deleting existing database file if it exists
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

# Create the database and the table
conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE cities(city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [8]:
# Populate our database
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("INSERT OR REPLACE INTO cities(city_name, round_trip_price) VALUES(?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

# Some cities:
save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

In [9]:
# Method to get price for a city
def get_city_price(city_name: str) -> float | None:
    """Get the roundtrip ticket price to travel to the city."""
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

In [12]:
get_city_price("Madrid")

550.0

In [13]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=model_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [14]:
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_X43SZKLvscoKQKVJr4yj1afU', arguments='{"city_name":"London"}', name='get_city_price')]
[FunctionExecutionResult(content='299.0', name='get_city_price', call_id='call_X43SZKLvscoKQKVJr4yj1afU', is_error=False)]


"A roundtrip to London will cost you $299! That's a bargain, considering you can't even buy a decent cup of tea in London for that price! Shall I book your ticket?"